# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [3]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [4]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [6]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

In [7]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")
df_de = pd.read_pickle("../data/dataset_de.pkl")

print("EU dataset:", df.shape)
print("DE dataset:", df_de.shape)

EU dataset: (4039906, 75)
DE dataset: (303349, 75)


--------------
### Apply preprocessing pipeline
-----------

In [8]:
# ---------------------------------------------------------
# full EU dataset
# ---------------------------------------------------------

df_clean = preprocess(df)


In [9]:
# shape of the cleaned EU dataset
print(df_clean.shape)

(4039906, 34)


In [10]:
# inspekt of the cleaned EU dataset
overview(df_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,4039906,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,4039906,0,0.00,7,"[3, 6, 18, 25, 21, 23, 22]"
DT_DISPATCH,datetime64[us],4039906,0,0.00,3296,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."
XSD_VERSION,str,4039906,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,4039906,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,4039906,0,0.00,10,"[0, 1, 2, 3, 5, 4, 84, 7, 8, 6]"
ISO_COUNTRY_CODE,str,4039906,0,0.00,33,"[DE, FR, ES, SE, PL, IT, HU, CY, UK, RO, PT, N..."
CAE_TYPE,str,4039906,0,0.00,10,"[8, 3, 1, 6, R, N, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,str,4039906,0,0.00,3,"[Unknown, Y, N]"
TYPE_OF_CONTRACT,str,4039906,0,0.00,3,"[W, U, S]"


In [11]:
# ---------------------------------------------------------
# Germany
# ---------------------------------------------------------

df_de_clean = preprocess(df_de)

In [12]:
# shape of the cleaned dataset for Germany
print(df_de_clean.shape)

(303349, 34)


In [13]:
# inspekt of the cleaned dataset  for Germany
overview(df_de_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,303349,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,303349,0,0.00,5,"[3, 6, 18, 25, 21]"
DT_DISPATCH,datetime64[us],303349,0,0.00,2803,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."
XSD_VERSION,str,303349,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,303349,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,303349,0,0.00,3,"[0, 1, 2]"
ISO_COUNTRY_CODE,str,303349,0,0.00,1,[DE]
CAE_TYPE,str,303349,0,0.00,10,"[8, 3, 6, N, 1, R, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,str,303349,0,0.00,3,"[Unknown, Y, N]"
TYPE_OF_CONTRACT,str,303349,0,0.00,3,"[W, U, S]"


#### Notes: Summary of Data Cleaning Results

1. Dataset Size After Preprocessing
- Full EU dataset:
  - Before: 4,039,906 rows × 75 columns
  - After: 4,039,906 rows × 34 columns

- Germany-only dataset:
  - Before: 303,349 rows × 75 columns
  - After: 303,349 rows × 34 columns 

2. The reduction from 75 to 34 columns is the result of removing:
- Columns removed due to >40% missing values (with the exception of "TITLE" and "CRIT_CRITERIA", which are important for NLP)
  - Winner information (WIN_*)
  - Contracting authority details (CAE_*)
  - GPA-related fields
  - Secondary financial fields (VALUE_EURO_FIN_*, AWARD_VALUE_EURO_FIN_1)
  - Award criteria weights (CRIT_*)
  - Additional CPVs
  - High-cardinality procedural flags (B_MULTIPLE_, B_FRA_, FRA_ESTIMATED, etc.)
  - TED_NOTICE_URL

- Columns removed due to irrelevance for competition modelling
  - Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
  - Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
  - Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)

3. Key Variables Retained Despite Missing Values
Several columns with substantial missingness were intentionally retained because they are critical for modelling tender competition and failure risk:
- VALUE_EURO
  - Missing: 39.85% (EU dataset)
  - Missing: 50.28% (Germany)
  - Importance: baseline contract value, used for log-transformations and value bins.

- AWARD_VALUE_EURO
  - Missing: 27.56% (EU dataset)
  - Missing: 43.69% (Germany)
  - Importance: actual awarded value, essential for understanding tender dynamics.

- TITLE
  - Missing: 49.40% (EU dataset)
  - Missing: 43.69% (Germany)
  - Importance: actual awarded value, essential for understanding tender dynamics.

- CRIT_PRICE_WEIGHT, CRIT_CRITERIA, CRIT_WEIGHTS
  - Missing: 60.85%, 54.05%,	56.69% (EU dataset)
  - Missing: 60.09%,	56.07%,	58.14% (Germany)
  - Importance: capture how a tender is evaluated—its balance of price vs. quality, the complexity of requirements, and the structure of scoring—making them essential indicators of competitiveness and failure risk.

These variables were not removed because:
They are central to the analytical goal (competition modelling).
Missingness is informative, not random.

4. Additional Preprocessing Steps
- All categorical missing values were replaced with "Unknown".
- Numeric missing values were left as NaN, to be handled during modelling.

5. Conversion of Categorical Variables
To ensure correct feature engineering and avoid dtype-related errors, all categorical variables were explicitly converted to string (object).
This includes fields such as:
  - ISO_COUNTRY_CODE
  - CAE_TYPE
  - TYPE_OF_CONTRACT
  - TAL_LOCATION_NUTS
  - B_EU_FUNDS
  - TOP_TYPE
  - CRIT_CODE
  - B_ELECTRONIC_AUCTION
  - B_AWARDED_TO_A_GROUP
  - WIN_COUNTRY_CODE
  - B_CONTRACTOR_SME
  - B_SUBCONTRACTED
  - CPV (converted from float → string, with removal of “.0”)
This guarantees correct handling of categorical data and enables creation of CPV hierarchy features in the feature engineering stage.

--------------
### Save cleaned datasets

-----------

In [14]:
df_clean.to_pickle("../data/dataset_clean.pkl")
df_de_clean.to_pickle("../data/dataset_de_clean.pkl")